In [14]:
from __future__ import annotations

import sys
from pathlib import Path
from pprint import pprint
from zipfile import ZipFile




In [15]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ASSET_DIR = PROJECT_ROOT / "notebooks" / "_generated_assets" / "05_multimodal_rag"
ASSET_DIR.mkdir(parents=True, exist_ok=True)

from src.rag import (
    MultimodalRetriever,
    chunk_document,
    chunk_documents,
    load_document,
    load_documents,
    prepare_rag_prompt,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Asset directory: {ASSET_DIR}")

Project root: C:\agentic\Property-Rental-Management-System
Asset directory: C:\agentic\Property-Rental-Management-System\notebooks\_generated_assets\05_multimodal_rag


In [16]:
def write_docx(path: Path, text: str) -> None:
    document_xml = (
        '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
        '<w:document xmlns:w="http://schemas.openxmlformats.org/wordprocessingml/2006/main">'
        "<w:body>"
        f"<w:p><w:r><w:t>{text}</w:t></w:r></w:p>"
        "</w:body>"
        "</w:document>"
    )

    with ZipFile(path, "w") as archive:
        archive.writestr("word/document.xml", document_xml)


def write_simple_pdf(path: Path, lines: list[str]) -> None:
    operations: list[str] = []
    y_position = 720

    for line in lines:
        safe_line = (
            line.replace("\\", "\\\\")
            .replace("(", "\\(")
            .replace(")", "\\)")
        )
        operations.extend([
            "BT",
            "/F1 12 Tf",
            f"72 {y_position} Td",
            f"({safe_line}) Tj",
            "ET",
        ])
        y_position -= 18

    stream = "\n".join(operations)
    pdf_bytes = (
        "%PDF-1.4\n"
        "1 0 obj\n<<>>\nstream\n"
        f"{stream}\n"
        "endstream\nendobj\n%%EOF"
    ).encode("latin-1")
    path.write_bytes(pdf_bytes)


def write_mock_png(path: Path) -> None:
    path.write_bytes(b"\x89PNG\r\n\x1a\nmock")


def preview_text(text: str, limit: int = 100) -> str:
    return text if len(text) <= limit else f"{text[: limit - 3]}..."


def describe_documents(documents):
    return [
        {
            "document_id": document.document_id,
            "modality": document.modality,
            "source": Path(document.source).name,
            "preview": preview_text(document.content),
            "metadata": document.metadata,
        }
        for document in documents
    ]


def describe_results(results):
    return [
        {
            "source": Path(result.chunk.source).name,
            "modality": result.chunk.modality,
            "score": result.score,
            "matched_terms": list(result.matched_terms),
            "text": result.chunk.text,
        }
        for result in results
    ]


In [17]:
lease_path = ASSET_DIR / "lease_unit_A1.txt"
invoice_path = ASSET_DIR / "july_rent_invoice.pdf"
maintenance_path = ASSET_DIR / "maintenance_request.docx"
parking_image_path = ASSET_DIR / "visitor_parking.png"
inspection_image_path = ASSET_DIR / "inspection_kitchen.png"

lease_path.write_text(
    " ".join(
        [
            "Lease summary for Unit A1.",
            "Tenants may use the rooftop garden between 08:00 and 20:00.",
            "Quiet hours begin at 22:00 and pets require written approval.",
        ]
    ),
    encoding="utf-8",
)

write_simple_pdf(
    invoice_path,
    [
        "July rent invoice for Unit A1",
        "Status: rent received in full on 2026-07-03.",
    ],
)

write_docx(
    maintenance_path,
    "Maintenance request for Unit A1: leaking kitchen tap approved. Technician visit scheduled for 2026-07-09 at 10:00.",
)

write_mock_png(parking_image_path)
write_mock_png(inspection_image_path)

image_descriptions = {
    str(parking_image_path): "Annotated site photo showing the visitor parking bays behind Building C beside the refuse area.",
    str(inspection_image_path): "Inspection photo shows ceiling water damage above the kitchen sink in Unit A1.",
}

metadata_by_source = {
    str(lease_path): {"category": "lease", "unit": "A1"},
    str(invoice_path): {"category": "finance", "unit": "A1", "month": "2026-07"},
    str(maintenance_path): {"category": "maintenance", "unit": "A1", "priority": "medium"},
    str(parking_image_path): {"category": "site-plan", "area": "parking"},
    str(inspection_image_path): {"category": "inspection", "unit": "A1", "priority": "high"},
}

single_image_document = load_document(
    inspection_image_path,
    description=image_descriptions[str(inspection_image_path)],
    metadata=metadata_by_source[str(inspection_image_path)],
)

documents = load_documents(
    [lease_path, invoice_path, maintenance_path, parking_image_path, inspection_image_path],
    descriptions=image_descriptions,
    metadata_by_source=metadata_by_source,
)

print("Single-document image example:")
pprint(describe_documents([single_image_document]))
print("\nMultimodal corpus:")
pprint(describe_documents(documents))


Single-document image example:
[{'document_id': 'inspection_kitchen',
  'metadata': {'category': 'inspection',
               'extension': '.png',
               'mime_type': 'image/png',
               'priority': 'high',
               'size_bytes': 12,
               'unit': 'A1'},
  'modality': 'image',
  'preview': 'Image asset inspection_kitchen.png. Inspection photo shows '
             'ceiling water damage above the kitchen...',
  'source': 'inspection_kitchen.png'}]

Multimodal corpus:
[{'document_id': 'lease_unit_A1',
  'metadata': {'category': 'lease',
               'extension': '.txt',
               'mime_type': 'text/plain',
               'size_bytes': 148,
               'unit': 'A1'},
  'modality': 'text',
  'preview': 'Lease summary for Unit A1. Tenants may use the rooftop garden '
             'between 08:00 and 20:00. Quiet hour...',
  'source': 'lease_unit_A1.txt'},
 {'document_id': 'july_rent_invoice',
  'metadata': {'category': 'finance',
               'extens

In [18]:
lease_chunks = chunk_document(documents[0], chunk_size=12, chunk_overlap=3)
all_chunks = chunk_documents(documents, chunk_size=12, chunk_overlap=3)

print(f"Lease chunk count: {len(lease_chunks)}")
pprint(
    [
        {
            "chunk_id": chunk.chunk_id,
            "text": chunk.text,
            "metadata": chunk.metadata,
        }
        for chunk in lease_chunks
    ]
)

print(f"\nTotal chunk count across the corpus: {len(all_chunks)}")


Lease chunk count: 3
[{'chunk_id': 'lease_unit_A1-chunk-0',
  'metadata': {'category': 'lease',
               'chunk_index': 0,
               'extension': '.txt',
               'mime_type': 'text/plain',
               'size_bytes': 148,
               'unit': 'A1',
               'word_end': 12,
               'word_start': 0},
  'text': 'Lease summary for Unit A1. Tenants may use the rooftop garden '
          'between'},
 {'chunk_id': 'lease_unit_A1-chunk-1',
  'metadata': {'category': 'lease',
               'chunk_index': 1,
               'extension': '.txt',
               'mime_type': 'text/plain',
               'size_bytes': 148,
               'unit': 'A1',
               'word_end': 21,
               'word_start': 9},
  'text': 'rooftop garden between 08:00 and 20:00. Quiet hours begin at 22:00 '
          'and'},
 {'chunk_id': 'lease_unit_A1-chunk-2',
  'metadata': {'category': 'lease',
               'chunk_index': 2,
               'extension': '.txt',
              

In [19]:
retriever = MultimodalRetriever()
retriever.add_documents(documents, chunk_size=12, chunk_overlap=3)

search_examples = {
    "garden_policy": describe_results(
        retriever.search("When can tenants use the rooftop garden?", top_k=2)
    ),
    "visitor_parking_image": describe_results(
        retriever.search("Where is the visitor parking?", top_k=2, modality="image")
    ),
    "paid_rent_pdf": describe_results(
        retriever.search(
            "Which unit already paid rent in full?",
            top_k=2,
            modality="pdf",
            filters={"category": "finance"},
        )
    ),
    "inspection_damage": describe_results(
        retriever.search(
            "What damage was found in Unit A1?",
            top_k=2,
            modality="image",
            filters={"category": "inspection"},
        )
    ),
    "maintenance_schedule": describe_results(
        retriever.search(
            "When is the kitchen tap repair scheduled?",
            top_k=2,
            modality="word",
            filters={"category": "maintenance"},
        )
    ),
}

pprint(search_examples)

print("\nContext block for a parking question:\n")
print(retriever.build_context("Where is the visitor parking?", top_k=2, modality="image"))


{'garden_policy': [{'matched_terms': ['garden',
                                      'rooftop',
                                      'tenants',
                                      'the',
                                      'use'],
                    'modality': 'text',
                    'score': 1.004105,
                    'source': 'lease_unit_A1.txt',
                    'text': 'Lease summary for Unit A1. Tenants may use the '
                            'rooftop garden between'},
                   {'matched_terms': ['garden', 'rooftop'],
                    'modality': 'text',
                    'score': 0.318173,
                    'source': 'lease_unit_A1.txt',
                    'text': 'rooftop garden between 08:00 and 20:00. Quiet '
                            'hours begin at 22:00 and'}],
 'inspection_damage': [{'matched_terms': ['a1', 'in', 'unit'],
                        'modality': 'image',
                        'score': 0.83565,
                        '

In [20]:
question = "What do the documents say about Unit A1 rent, maintenance, and inspection issues?"
prompt = prepare_rag_prompt(
    question,
    retriever,
    top_k=4,
    filters={"unit": "A1"},
)

print(prompt)


Use the retrieved context to answer the question. Cite the relevant context blocks by their bracketed numbers.

Question: What do the documents say about Unit A1 rent, maintenance, and inspection issues?

Context:
[1] C:\agentic\Property-Rental-Management-System\notebooks\_generated_assets\05_multimodal_rag\july_rent_invoice.pdf (pdf)
July rent invoice for Unit A1 Status: rent received in full on

[2] C:\agentic\Property-Rental-Management-System\notebooks\_generated_assets\05_multimodal_rag\inspection_kitchen.png (image)
above the kitchen sink in Unit A1.

[3] C:\agentic\Property-Rental-Management-System\notebooks\_generated_assets\05_multimodal_rag\maintenance_request.docx (word)
Maintenance request for Unit A1: leaking kitchen tap approved. Technician visit scheduled

[4] C:\agentic\Property-Rental-Management-System\notebooks\_generated_assets\05_multimodal_rag\inspection_kitchen.png (image)
Image asset inspection_kitchen.png. Inspection photo shows ceiling water damage above the kit